In [40]:
from dotenv import load_dotenv
load_dotenv()

True

In [41]:
# even tho I am not getting married. 
# we building a wedding planner agent. 
# 
# the agent is supposed to use mcp servers, web search, and multi agents. 
# 
# 3 subagents and 1 main agent
# 
# 1 subagent to search for flights using the kiwi MCP server
# 1 subagent to search the web for venue details using the Tavily API
# 1 subagent to create a music playlist (maybe use a JSON db again with some generated music items)
# 1 main agent to update state and use the subagents to plan the entire wedding. 

In [42]:
# step 1: travel agent
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "flight_server": {
        "transport": "streamable_http",
        "url": "https://mcp.kiwi.com"
    }
})

tools = await client.get_tools()

In [43]:
from langchain.agents import create_agent

flight_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=tools,
    system_prompt="""You are a travel agent. Your role is to search for flights to the wedding location. Do not ask for extra context or questions. Search for flights based on the shortest distance, best date to visit the location and the price (lowest, economy class or premium economy). Let the user know of the best options that you have shortlisted after critical analysis.

IMPORTANT: When calling the search-flight tool, you MUST use a departure date that is in the future. The API rejects past or today's date. Use the date or month given by the user (e.g. October 2026 -> use 2026-10-01 or another day in that month). Always pass departureDate in YYYY-MM-DD format."""
)

In [44]:
# step 2: venue agent
from tavily import TavilyClient
from typing import Dict, Any
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def search_web(query: str) -> Dict[str, Any]:
    """search the web for information."""
    return tavily_client.search(query)

In [45]:
from langchain.agents import create_agent

venue_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_web],
    system_prompt="""You are a expert in finding venues. Your role is to search for venues in the desired wedding location with the desired capacity of guests. Do not ask any follow up questions, only find the venue with the desired options following price (lowest), capacity (exact match), and reviews (highest). Make iterative searches if required to get the best options."""
)

In [ ]:
# step 3: music playlist agent
from langchain.tools import tool
import json

@tool
def get_music() -> dict:
    """gets the music from JSON dictionary"""
    with open('../resources/music.json', 'r') as file: 
        data = json.load(file)
    
    return data['music_database']

In [47]:
from langchain.agents import create_agent

music_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[get_music],
    system_prompt="""You are a DJ. Your role is to use the tool retreive the list of music and curate a playlist for the wedding based on the location of the wedding. Once the playlist is ready, calculate the toatl duration and cost of the playlist. Do iterative tries to find the best playlist and return it to the user."""
)

In [ ]:
# step 4: create custom state
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guests: str
    wedding_date: str = ""  # e.g. "October 2026" - needed so flight search uses a future date for Kiwi -- cursor added to fix error. 

In [ ]:
# step 5: create tools for main agent
import asyncio
import time
from langchain.tools import ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

#################### Cursor FIX BLOCK - TO BE REVIEWED ##########################
# Rate limit: 30k input tokens/min -> ~2 LLM calls/min. Serialize specialist calls with 30s spacing.
MIN_INTERVAL_BETWEEN_LLM_CALLS = 30
_last_llm_time = 0.0
_llm_lock = asyncio.Lock()

async def _rate_limit_llm():
    global _last_llm_time
    async with _llm_lock:
        now = time.monotonic()
        wait = _last_llm_time + MIN_INTERVAL_BETWEEN_LLM_CALLS - now
        if wait > 0:
            await asyncio.sleep(wait)
        _last_llm_time = time.monotonic()
#################### Cursor FIX BLOCK - TO BE REVIEWED -- END #######################

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights from origin to destination. Uses wedding_date from state so Kiwi gets a future departure date."""
    await _rate_limit_llm()
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    wedding_date = runtime.state.get("wedding_date") or "next month"
    prompt = f"find flights from {origin} to {destination} for {wedding_date}. Use a departure date that is in the future in YYYY-MM-DD format."
    response = await flight_agent.ainvoke(
        {"messages": [{"role": "user", "content": prompt}]}
    )
    return response['messages'][-1].content

@tool
async def search_venue(runtime: ToolRuntime) -> str: 
    """Venue expert searches and chooses the best venue based on the wedding location and guests"""
    await _rate_limit_llm()
    destination = runtime.state["destination"]
    guests = runtime.state["guests"]
    response = await venue_agent.ainvoke(
        {"messages": [{"role": "user", "content": f"find wedding venues in {destination} for {guests} guests"}]}
    )
    return response['messages'][-1].content

@tool
async def playlist(runtime: ToolRuntime) -> str:
    """DJ agent curates a playlist for the wedding based on the destination."""
    await _rate_limit_llm()
    destination = runtime.state["destination"]
    response = await music_agent.ainvoke(
        {"messages": [{"role": "user", "content": f"curate the best playlist from the music db for the {destination}"}]}
    )
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guests: str, wedding_date: str = "", runtime: ToolRuntime = None) -> str:
    """Update the state when all values are revealed: origin, destination, guests, and optionally wedding_date (e.g. October 2026). Wedding date is needed so flight search uses a future date."""
    update = {
        "origin": origin,
        "destination": destination,
        "guests": guests,
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]
    }
    if wedding_date:
        update["wedding_date"] = wedding_date
    return Command[tuple[()]](update=update)

In [50]:
# step 6: create main agent
from langchain.agents import create_agent

main_agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[search_flights, search_venue, playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""You are the best wedding planner. Your role is to delegate the tasks to your specialists. Search flights, venues and curate playlists. Gather all information and update the state. When updating state, always pass the wedding date (e.g. October 2026) if the user mentioned it - this is required so flight search uses a future date. Once the state is updated assign the tasks to the specialists. Get their expert answers and plan the best wedding ever."""
)

In [51]:
# step 7: invoke the main agent (specialist tools are rate-limited to stay under 30k input tokens/min)
response = await main_agent.ainvoke(
    {"messages": [{"role": "user", "content": "I am based in UAE and I would like to do wedding in Mumbai for 200 guests in October 2026"}]}
)

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [ ]:
# This is proving to be an expensive multi agent setup to run.
# will revisit this later. 

In [ ]:
print(response)

In [ ]:
print(response['messages'][-1].content)